<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        K-Means y Métodos Particionales
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Programación para Ciencia de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Módulo 10
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Data%20Science%20programming/10%20-%20Clustering/01_KMeans_y_Metodos_Particionales.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os, urllib.request
import warnings
warnings.filterwarnings('ignore')

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (8.5, 4.5)
plt.rcParams['font.size'] = 10

def load_dataset(filename, module_folder="10 - Clustering"):
    local_path = os.path.join(os.getcwd(), "data", filename)
    if os.path.exists(local_path):
        return local_path
    
    parent_path = os.path.join(os.getcwd(), "..", module_folder, "data", filename)
    if os.path.exists(parent_path):
        return parent_path

    raw_url = f"https://raw.githubusercontent.com/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/main/Data%20Science%20programming/{module_folder.replace(' ', '%20')}/data/{filename}"
    os.makedirs("data", exist_ok=True)
    target_path = os.path.join("data", filename)
    if not os.path.exists(target_path):
        urllib.request.urlretrieve(raw_url, target_path)
    return target_path

print("🚀 Entorno configurado exitosamente para el Módulo 10: Clustering.")


---
### 1. El Algoritmo K-Means (Algoritmo de Lloyd) 🔄

**K-Means** es el algoritmo de agrupamiento particional más popular del mundo. Dado un número fijo de clusters $k$, el algoritmo particiona el conjunto de $n$ observaciones en $k$ conjuntos $S = \{S_1, S_2, \dots, S_k\}$ minimizando la suma de distancias cuadradas al centroide más cercano:

$$rg\min_S \sum_{j=1}^k \sum_{x_i \in S_j} \|x_i - \mu_j\|^2$$

donde $\mu_j$ es la media o **centroide** del cluster $S_j$.

#### Pasos Iterativos de Optimización (Expectation-Maximization):
1. **Inicialización:** Se eligen $k$ centroides iniciales $\mu_1^{(0)}, \dots, \mu_k^{(0)}$.
2. **Paso de Asignación (Expectation):** Cada observación se asigna al centroide más cercano según la distancia Euclidiana:
   $$S_j^{(t)} = \left\{ x_i : \|x_i - \mu_j^{(t)}\|^2 \le \|x_i - \mu_{l}^{(t)}\|^2 \; orall l \in \{1, \dots, k\} ight\}$$
3. **Paso de Actualización (Maximization):** Se recalculan los nuevos centroides como el promedio vectorial de todos los puntos asignados a cada cluster:
   $$\mu_j^{(t+1)} = rac{1}{|S_j^{(t)}|} \sum_{x_i \in S_j^{(t)}} x_i$$
4. **Criterio de Parada:** El algoritmo finaliza cuando los centroides convergen ($\|\mu^{(t+1)} - \mu^{(t)}\| < \epsilon$) o se alcanza el número máximo de iteraciones (`max_iter`).


---
### 2. Inicialización Inteligente con K-Means++ 🎯

> [!IMPORTANT]
> El K-Means clásico con inicialización puramente aleatoria puede converger a **mínimos locales desfavorables** dependiendo de la posición de los centroides iniciales.
> 
> **K-Means++** soluciona esto dispersando los centroides iniciales mediante una distribución de probabilidad proporcional al cuadrado de la distancia al centroide más cercano:
> $$P(x) = rac{D(x)^2}{\sum_{x' \in X} D(x')^2}$$
> Esto garantiza una convergencia más rápida y una cota teórica de optimalidad $\mathcal{O}(\log k)$.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Cargar datos de clientes
path_data = load_dataset('mall_customers.csv', '10 - Clustering')
df = pd.read_csv(path_data)

X = df[['Annual_Income_k', 'Spending_Score']].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Ajustar K-Means con k=5 y K-Means++
kmeans = KMeans(n_clusters=5, init='k-means++', n_init=10, random_state=42)
cluster_labels = kmeans.fit_predict(X_scaled)
df['Cluster'] = cluster_labels

print("Inercia final del modelo (WCSS):", kmeans.inertia_)
print("Coordenadas de los 5 centroides (escaladas):\n", kmeans.cluster_centers_)


---
### 3. Visualización de Clusters y Fronteras de Voronoi 🗺️


In [ ]:
plt.figure(figsize=(9, 5))
colores = ['#0284c7', '#10b981', '#f59e0b', '#ef4444', '#8b5cf6']
nombres = ['Ingreso Bajo / Gasto Bajo', 'Ingreso Alto / Gasto Alto (VIP)', 
           'Ingreso Medio / Gasto Medio', 'Ingreso Bajo / Gasto Alto', 'Ingreso Alto / Gasto Bajo']

for i in range(5):
    plt.scatter(X[cluster_labels == i, 0], X[cluster_labels == i, 1], 
                s=60, c=colores[i], label=f'Cluster {i}: {nombres[i]}', alpha=0.75, edgecolors='none')

# Des-escalar centroides para graficar en el espacio original
centroides_originales = scaler.inverse_transform(kmeans.cluster_centers_)
plt.scatter(centroides_originales[:, 0], centroides_originales[:, 1], 
            s=220, c='#0f172a', marker='X', edgecolors='white', linewidths=2, label='Centroides (μ)')

plt.title('Segmentación de Clientes con K-Means (k = 5)', fontsize=13, fontweight='bold')
plt.xlabel('Ingreso Anual (miles de USD)', fontweight='bold')
plt.ylabel('Puntuación de Gasto (1-100)', fontweight='bold')
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True)
plt.tight_layout()
plt.show()


---
### 4. MiniBatchKMeans para Grandes Volúmenes de Datos ⚡

Para conjuntos de datos con millones de filas, `MiniBatchKMeans` reduce drásticamente el tiempo de cálculo utilizando muestras aleatorias (*mini-batches*) en cada iteración:


In [ ]:
from sklearn.cluster import MiniBatchKMeans
import time

mb_kmeans = MiniBatchKMeans(n_clusters=5, batch_size=64, random_state=42, n_init=3)
mb_labels = mb_kmeans.fit_predict(X_scaled)

print("Inercia MiniBatchKMeans:", mb_kmeans.inertia_)
print("Diferencia de inercia vs K-Means estándar:", abs(kmeans.inertia_ - mb_kmeans.inertia_))


---
##### 🛠️ Práctica 2: Segmentación Multidimensional (Edad + Ingreso + Gasto)

**Reto:**
1. Construye una matriz $X_{\text{3D}}$ con las características `Age`, `Annual_Income_k` y `Spending_Score`.
2. Escala los datos con `StandardScaler`.
3. Ajusta un modelo `KMeans` con $k=4$ y semilla `random_state=42`.
4. Calcula el promedio de `Age`, `Annual_Income_k` y `Spending_Score` para cada uno de los 4 clusters resultantes y describe verbalmente el perfil de cada segmento.


In [ ]:
# =========================================================================
# TU SOLUCIÓN: Práctica 2 - Segmentación 3D con K-Means
# =========================================================================

# 1. Definir X_3D
# X_3D = df[['Age', 'Annual_Income_k', 'Spending_Score']].values

# 2. Escalar datos
# X_3D_scaled = StandardScaler().fit_transform(X_3D)

# 3. Ajustar KMeans con k=4
# km_3d = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
# df['Cluster_3D'] = km_3d.fit_predict(X_3D_scaled)

# 4. Tabla de perfiles promedio
# print(df.groupby('Cluster_3D')[['Age', 'Annual_Income_k', 'Spending_Score']].mean().round(2))


<details>
<summary><b>💡 Haz clic aquí para ver la Solución Paso a Paso y Explicación</b></summary>
<br>

```python
# 1. Definir matriz tridimensional
X_3D = df[['Age', 'Annual_Income_k', 'Spending_Score']].values

# 2. Escalar
X_3D_scaled = StandardScaler().fit_transform(X_3D)

# 3. Ajustar modelo K-Means
km_3d = KMeans(n_clusters=4, init='k-means++', n_init=10, random_state=42)
df['Cluster_3D'] = km_3d.fit_predict(X_3D_scaled)

# 4. Análisis de perfiles de clientes
perfiles = df.groupby('Cluster_3D')[['Age', 'Annual_Income_k', 'Spending_Score']].agg(['mean', 'count'])
print("--- Perfil Promedio de los 4 Clusters 3D ---")
print(df.groupby('Cluster_3D')[['Age', 'Annual_Income_k', 'Spending_Score']].mean().round(1))
```
</details>


---
### 6. Resumen y Conclusiones del Cuaderno 01 📌

1. **Optimización de Inercia:** K-Means minimiza la varianza intra-cluster asignando iterativamente puntos al centroide más próximo y recalculando medias.
2. **K-Means++:** Es el estándar obligatorio de inicialización para evitar mínimos locales y acelerar la convergencia.
3. **Fronteras Convexas:** K-Means asume implícitamente clusters esféricos e isotrópicos de tamaño similar; no es adecuado para geometrías alargadas o concéntricas.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Programación para Ciencia de Datos</i>
  </p>
</div>
